# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [2]:
# Initialize and constants

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [3]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 5/5 [01:32<00:00, 18.54s/it]


In [4]:
len(deals)

50

In [5]:
deals[44].describe()

'Title: Vevor 25.4" Built-In Grill Cabinet BBQ Drawers for $152 + free shipping\nDetails: The going rate on this elsewhere today is $190 or more. Buy Now at Lowe\'s\nFeatures: \nURL: https://www.dealnews.com/Vevor-25-4-Built-In-Grill-Cabinet-BBQ-Drawers-for-152-free-shipping/21733593.html?iref=rss-c196'

## Use LLM to scrape the price.

Examp;le of rss item:

<item>
<title>Best Buy Outlet Event TV Deals from $70 + free shipping</title>
<link>https://www.dealnews.com/Best-Buy-Outlet-Event-TV-Deals-from-70-free-shipping/21732323.html?iref=rss-c142</link>
<description><img src='https://c.dlnws.com/image/upload/f_auto,t_large,q_auto/cms/ebssqfbxmn4vek5f0ptm.jpg' style='float: left;vertical-align: top;margin: 0 8px 8px 0'><div class="snippet summary" title="There&#x20;are&#x20;nearly&#x20;400&#x20;TVs&#x20;discounted&#x20;as&#x20;part&#x20;of&#x20;this&#x20;outlet&#x20;event.&#x20;You&#x20;will&#x20;save&#x20;on&#x20;new,&#x20;refurbished,&#x20;and&#x20;open-box&#x20;TVs&#x20;from&#x20;brands&#x20;like&#x20;Sony,&#x20;Samsung,&#x20;LG,&#x20;and&#x20;more.&#x20;Shipping&#x20;is&#x20;free&#x20;for&#x20;most&#x20;items,&#x20;but&#x20;some&#x20;are&#x20;available&#x20;for&#x20;store&#x20;pickup&#x20;only."> <p>There are nearly 400 TVs discounted as part of this outlet event. You will save on new, refurbished, and open-box TVs from brands like Sony, Samsung, LG, and more. Shipping is free for most items, but some are available for store pickup only. Buy Now at Best Buy </p> </div></description>
<guid>https://www.dealnews.com/21732323.html?iref=rss-c142</guid>
<pubDate>Sun, 27 Apr 2025 06:08:35 -0400</pubDate>
</item>


<b>The rss feed item (deals) don't have a price attribute. So the price needs to be extracted from the description. Sometimes it's difficult because there is a 20% discount applied to it... etc So we will ask a Frontier model to extract the price for us

In [26]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [18]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

We shouldn't need to specify the json format because we already have the Structure Format. But it's for enforcing.

In [27]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [28]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Best Buy Outlet Event TV Deals from $70 + free shipping
Details: There are nearly 400 TVs discounted as part of this outlet event. You will save on new, refurbished, and open-box TVs from brands like Sony, Samsung, LG, and more. Shipping is free for most items, but some are available for store pickup only. Buy Now at Best Buy
Features: 
URL: https://www.dealnews.com

## Call Frontier model with a response format from pydantic

In [29]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection  # Our class as format
    )
    result = completion.choices[0].message.parsed
    return result

In [30]:
result = get_recommendations()

In [31]:
type(result)

agents.deals.DealSelection

In [32]:
len(result.deals)

5

In [33]:
result.deals[1]

Deal(product_description='The Kospet Tank T2 Smartwatch is designed for durability and functionality, featuring a robust silicone and steel strap that is suitable for various sports activities. With a vibrant 1.43" AMOLED display, this smartwatch boasts a remarkable battery life of over 50 days on standby and around 10 to 15 days with regular use. It supports up to 70 sports modes, making it ideal for fitness enthusiasts, and includes advanced heart and blood pressure monitoring technologies, ensuring users stay on top of their health metrics.', price=59.99, url='https://www.dealnews.com/Kospet-Tank-T2-Smartwatch-Clearance-Sale-for-60-free-shipping/21733599.html?iref=rss-c142')

In [16]:
completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection  # Our class as format
    )
result2 = completion.choices[0].message

In [17]:
result2

ParsedChatCompletionMessage[DealSelection](content='{\n  "deals": [\n    {\n      "product_description": "The Kospet Tank T2 Smartwatch is designed for durability and performance. It features a 1.43-inch AMOLED display that delivers vibrant colors and sharp details. With an impressive battery life that supports over 50 days on standby and 10 to 15 days with regular use, this smartwatch is perfect for anyone who wants reliable performance. Its IP69K water rating and support for up to 70 sports modes, including hiking and boxing, make it an ideal choice for active individuals seeking health and fitness monitoring.",\n      "price": 60,\n      "url": "https://www.dealnews.com/Kospet-Tank-T2-Smartwatch-Clearance-Sale-for-60-free-shipping/21733599.html?iref=rss-c142"\n    },\n    {\n      "product_description": "The Unlocked Samsung Galaxy S25 Ultra Android Phone offers a powerhouse of features in a sleek design. With options for both 512GB and 1TB storage, you can enjoy ample space for you

# Put that into an Agent

In [34]:
from agents.scanner_agent import ScannerAgent

In [35]:
agent = ScannerAgent()
result = agent.scan()

In [36]:
result

DealSelection(deals=[Deal(product_description="The Kospet Tank T2 Smartwatch features a vibrant 1.43-inch AMOLED display and boasts an impressive battery life, offering over 50 days of standby time and 10 to 15 days for daily usage. Designed with an IP69K water rating, it's perfect for those who lead an active lifestyle and supports up to 70 sports modes including hiking, skiing, and boxing. The smartwatch also includes vital health monitoring functions for tracking heart rate and blood pressure, making it a stylish and practical companion for health enthusiasts.", price=59.99, url='https://www.dealnews.com/Kospet-Tank-T2-Smartwatch-Clearance-Sale-for-60-free-shipping/21733599.html?iref=rss-c142'), Deal(product_description='The Unlocked Samsung Galaxy S25 Ultra Android Phone offers you an exceptional mobile experience with 512GB of storage available for the price of the 256GB model. This device features a stunning display and powerful performance, allowing you to multitask seamlessly a